In [2]:
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
!pip install -q opencv-python pycocotools matplotlib torch torchvision pillow

In [3]:
import sagemaker


sess = sagemaker.Session()
role_arn = sess.get_caller_identity_arn()
print(f"✓ IAM Execution Role ARN: {role_arn}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
✓ IAM Execution Role ARN: arn:aws:iam::861276118242:role/service-role/AmazonSageMaker-ExecutionRole-20260511T121821


In [4]:
import boto3
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.patheffects as pe 
import cv2
from PIL import Image
import io
import os
import tempfile
import json
from datetime import datetime
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

In [5]:
S3_BUCKET = 'ztm-sagemaker-course'
S3_IMAGE_KEY = ''
S3_MODEL_BUCKET = 'ztm-sagemaker-course'
S3_MODEL_KEY = 'sam_vit_h_4b8939.pth'
MODEL_TYPE = 'vit_h'
S3_BUCKET_REGION = 'us-west-2'
OUTPUT_PREFIX = 'segmentation_output'

In [6]:
# FILTER: Only process these labels
TARGET_LABELS = {'Car', 'Vehicle', 'Automobile', 'Transportation', 'Person', 'Pedestrian', 'Human'}

In [7]:
# Initialize AWS clients
s3_client = boto3.client('s3', region_name=S3_BUCKET_REGION)
rekognition_client = boto3.client('rekognition', region_name=S3_BUCKET_REGION)

In [8]:
print("✓ Configuration loaded")
print(f"Image: s3://{S3_BUCKET}/{S3_IMAGE_KEY}")
print(f"Model: s3://{S3_MODEL_BUCKET}/{S3_MODEL_KEY}")
print(f"Target labels: {TARGET_LABELS}")

✓ Configuration loaded
Image: s3://ztm-sagemaker-course/
Model: s3://ztm-sagemaker-course/sam_vit_h_4b8939.pth
Target labels: {'Vehicle', 'Transportation', 'Person', 'Car', 'Pedestrian', 'Automobile', 'Human'}


In [9]:
# Helper functions

def load_model_from_s3(bucket, key):
    """Download SAM model from S3 to temporary location"""
    print(f"Loading SAM model from s3://{bucket}/{key}")
    temp_file = tempfile.NamedTemporaryFile(delete=False, suffix='.pth')
    temp_path = temp_file.name
    temp_file.close()

    print("Downloading model...")
    s3_client.download_file(bucket, key, temp_path)
    print(f"✓ Model downloaded")
    return temp_path

def get_image_from_s3(bucket, key, max_size=512):
    """Load image from S3 and convert to RGB"""
    response = s3_client.get_object(Bucket=bucket, Key=key)
    img_data = response['Body'].read()
    img = Image.open(io.BytesIO(img_data))

    if img.mode != 'RGB':
        print(f"  Converting image from {img.mode} to RGB")
        img = img.convert('RGB')

    # Resize if too large (for faster CPU processing)
    if max_size and max(img.size) > max_size:
        ratio = max_size / max(img.size)
        new_size = (int(img.size[0] * ratio), int(img.size[1] * ratio))
        print(f"  Resizing from {img.size} to {new_size} for faster processing")
        img = img.resize(new_size, Image.Resampling.LANCZOS)

    return img

def get_image_size(bucket, key):
    """Get image dimensions from S3"""
    response = s3_client.get_object(Bucket=bucket, Key=key)
    image_data = response['Body'].read()
    image = Image.open(io.BytesIO(image_data))
    if image.mode != 'RGB':
        image = image.convert('RGB')
    return image.size

def euclidean_distance(box1, box2):
    """Calculate Euclidean distance between two bounding boxes"""
    return np.sqrt(sum((a - b) ** 2 for a, b in zip(box1, box2)))

print("✓ Helper functions loaded")

✓ Helper functions loaded


In [10]:
def get_rekognition_detections(bucket, key, target_labels, min_confidence=95):
    """Get object detections from AWS Rekognition - FILTERED by target labels"""
    print(f"Calling AWS Rekognition...")

    response = rekognition_client.detect_labels(
        Image={'S3Object': {'Bucket': bucket, 'Name': key}},
        MaxLabels=100,
        MinConfidence=min_confidence
    )

    width, height = get_image_size(bucket, key)
    print(f"Image dimensions: {width}x{height}")

    # Extract and filter detections
    detections = []
    for label in response['Labels']:
        if label['Name'] in target_labels:
            for instance in label.get('Instances', []):
                box = instance['BoundingBox']
                x = int(box['Left'] * width)
                y = int(box['Top'] * height)
                w = int(box['Width'] * width)
                h = int(box['Height'] * height)

                detections.append({
                    'label': label['Name'],
                    'confidence': instance['Confidence'],
                    'bbox': [x, y, w, h]
                })

    unique_detections = []
    seen_boxes = set()
    for det in detections:
        box_tuple = tuple(det['bbox'])
        if box_tuple not in seen_boxes:
            seen_boxes.add(box_tuple)
            unique_detections.append(det)

    print(f"✓ Found {len(unique_detections)} target objects (cars/pedestrians)")
    return unique_detections

print("✓ Rekognition function loaded")

✓ Rekognition function loaded


In [ ]:
print("\n" + "="*50)
print("INITIALIZING SAM MODEL")
print("="*50)

sam_model_path = load_model_from_s3(S3_MODEL_BUCKET, S3_MODEL_KEY)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

if device == "cpu":
    print("⚠ Running on CPU - using memory-efficient settings")
    torch.set_num_threads(1)

try:
    print(f"Loading SAM model ({MODEL_TYPE})...")
    sam = sam_model_registry[MODEL_TYPE](checkpoint=sam_model_path)
    sam.to(device=device)

    mask_generator = SamAutomaticMaskGenerator(
        sam,
        points_per_side=32,
        pred_iou_thresh=0.88,
        stability_score_thresh=0.95,
        crop_n_layers=1,
        crop_n_points_downscale_factor=2,
        min_mask_region_area=100,
    )

    print("✓ SAM model loaded successfully!")
    os.unlink(sam_model_path)
    print("✓ Temporary model file cleaned up")

except RuntimeError as e:
    print(f"❌ Error: {e}")
    os.unlink(sam_model_path)
    raise

print("="*50)

In [11]:
def process_image(bucket, key, target_labels, resize_for_cpu=True, max_size=512):
    """Process image and return only cars and pedestrians with segmentation masks"""
    print("\n" + "="*50)
    print("PROCESSING IMAGE")
    print("="*50)

    print("Step 1: Loading image...")
    if resize_for_cpu and device == "cpu":
        image = get_image_from_s3(bucket, key, max_size=max_size)
    else:
        image = get_image_from_s3(bucket, key, max_size=None)

    image_np = np.array(image)
    print(f"✓ Image loaded: {image_np.shape}")

    print("\nStep 2: Getting Rekognition detections (cars/pedestrians only)...")
    rek_detections = get_rekognition_detections(bucket, key, target_labels)

    if len(rek_detections) == 0:
        print("⚠ No target objects (cars/pedestrians) detected")
        return image_np, [], []

    if resize_for_cpu and device == "cpu":
        original_img = Image.open(io.BytesIO(s3_client.get_object(Bucket=bucket, Key=key)['Body'].read()))
        if original_img.mode != 'RGB':
            original_img = original_img.convert('RGB')
        scale_x = image_np.shape[1] / original_img.size[0]
        scale_y = image_np.shape[0] / original_img.size[1]

        for det in rek_detections:
            x, y, w, h = det['bbox']
            det['bbox'] = [int(x*scale_x), int(y*scale_y), int(w*scale_x), int(h*scale_y)]

    print("\nStep 3: Generating SAM segmentation masks...")
    if device == "cpu":
        print(f"⚠ Running on CPU - will take 2-5 minutes")

    sam_masks = mask_generator.generate(image_np)
    print(f"✓ Generated {len(sam_masks)} SAM masks")

    print("\nStep 4: Matching target objects with SAM masks...")
    sam_bboxes = [(i, mask['bbox']) for i, mask in enumerate(sam_masks)]

    matched_results = []
    for rek_det in rek_detections:
        rek_bbox = rek_det['bbox']
        min_distance = float('inf')
        best_match = None

        for sam_idx, sam_bbox in sam_bboxes:
            distance = euclidean_distance(rek_bbox, sam_bbox)
            if distance < min_distance:
                min_distance = distance
                best_match = sam_idx

        if best_match is not None:
            matched_results.append({
                'label': rek_det['label'],
                'confidence': rek_det['confidence'],
                'rekognition_bbox': rek_bbox,
                'sam_bbox': sam_bboxes[best_match][1],
                'sam_mask_index': best_match,
                'segmentation_mask': sam_masks[best_match]['segmentation'],
                'area': sam_masks[best_match]['area']
            })

    print(f"✓ Matched {len(matched_results)} target objects")
    print("="*50)

    return image_np, sam_masks, matched_results

print("✓ Processing function loaded")

✓ Processing function loaded


In [ ]:
print("\n" + "#"*50)
print("PROCESSING PIPELINE")
print("#"*50)

image_np, all_sam_masks, matched_results = process_image(
    S3_BUCKET,
    S3_IMAGE_KEY,
    TARGET_LABELS,
    resize_for_cpu=True,
    max_size=512
)

print("\n" + "="*50)
print("DETECTION RESULTS (CARS & PEDESTRIANS ONLY)")
print("="*50)

if len(matched_results) == 0:
    print("No cars or pedestrians detected.")
else:
    for i, result in enumerate(matched_results, 1):
        print(f"\nObject {i}:")
        print(f"  Label:       {result['label']}")
        print(f"  Confidence:  {result['confidence']:.2f}%")
        print(f"  Mask Area:   {result['area']} pixels")

print("="*50)

In [ ]:
print("\n" + "="*50)
print("VISUALIZING STEP 2: REKOGNITION DETECTIONS")
print("="*50)

if len(matched_results) > 0:
    fig, ax = plt.subplots(1, 1, figsize=(15, 15))
    ax.imshow(image_np.copy())
    ax.set_autoscale_on(False)
    ax.set_title("AWS Rekognition: Detected Cars & People", fontsize=16, weight='bold')

    # Iterate through results to get labels and boxes
    for result in matched_results:
        x, y, w, h = result['rekognition_bbox']
        label_text = result['label']

        rect = patches.Rectangle((x, y), w, h, linewidth=3,
                                 edgecolor='cyan', facecolor='none', linestyle='--')
        ax.add_patch(rect)
        ax.text(x, y - 5, f"Rekognition: {label_text}", color='cyan',
                fontsize=11, weight='bold',
                bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))

    ax.axis('off')
    plt.tight_layout()
    plt.show()
    print("✓ Rekognition visualization complete.")
else:
    print("No target objects were detected by Rekognition to visualize.")